# 19 — Column Level Security (CLS) — vstone_catalog.security

## Step 1 — Confirm user identity and group membership

In [0]:
%sql

-- Column                        | Visible to              | Others get
-- latitude / longitude          | admin_group, safety_team | NULL (exact
--                                |                          | sensor/street
--                                |                          | coordinates
--                                |                          | withheld)
-- fact_citizen_reports.message  | admin_group, safety_team | **REDACTED**
--                                |                          | (free-text
--                                |                          | reports may
--                                |                          | contain
--                                |                          | identifying
--                                |                          | detail)
--
-- Adapted from the reference project's price/color CLS pattern to VStone's
-- own sensitive columns — VStone has no financial data, so the closest
-- analogous "sensitive" fields are exact geographic coordinates (precise
-- sensor locations, an operational-security concern) and free-text citizen
-- report content (potential PII in unstructured complaint text).
SELECT
  CURRENT_USER() AS current_user,
  IS_MEMBER('safety_team') AS is_safety,
  IS_MEMBER('admin_group') AS is_admin;

## Step 2 — CLS view on dim_street (coordinates masked)

In [0]:
%sql
CREATE OR REPLACE VIEW vstone_catalog.security.cls_dim_street AS
SELECT
  street_id,
  street_name,
  street_length_m,
  CASE WHEN IS_MEMBER('admin_group') OR IS_MEMBER('safety_team')
       THEN latitude ELSE NULL END AS latitude,
  CASE WHEN IS_MEMBER('admin_group') OR IS_MEMBER('safety_team')
       THEN longitude ELSE NULL END AS longitude,
  danger_score,
  __START_AT, __END_AT
FROM vstone_catalog.gold.dim_street;

##Step 3 — CLS view on dim_node_location (coordinates masked)

In [0]:
%sql
CREATE OR REPLACE VIEW vstone_catalog.security.cls_dim_node_location AS
SELECT
  location,
  CASE WHEN IS_MEMBER('admin_group') OR IS_MEMBER('safety_team')
       THEN latitude ELSE NULL END AS latitude,
  CASE WHEN IS_MEMBER('admin_group') OR IS_MEMBER('safety_team')
       THEN longitude ELSE NULL END AS longitude,
  has_valid_coordinates,
  __START_AT, __END_AT
FROM vstone_catalog.gold.dim_node_location;

## -- Step 4 — CLS view on fact_citizen_reports (message text redacted)

In [0]:
%sql
CREATE OR REPLACE VIEW vstone_catalog.security.cls_fact_citizen_reports AS
SELECT
  CAST(
    CASE WHEN IS_MEMBER('admin_group') OR IS_MEMBER('safety_team')
         THEN message ELSE '**REDACTED**' END
  AS STRING) AS message,
  message_date,
  message_hour,
  street_id,
  gold_load_dt
FROM vstone_catalog.gold.fact_citizen_reports;

-- Step 5 — Verify masked output
SELECT * FROM vstone_catalog.security.cls_dim_street LIMIT 5;
SELECT * FROM vstone_catalog.security.cls_fact_citizen_reports LIMIT 5;